<a href="https://colab.research.google.com/github/Snehamn24/RAG-GENAI/blob/main/RAG_GENAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-openai chromadb tiktoken sentence-transformers

In [ ]:
!pip install requests==2.32.4 --force-reinstall

Setting up API key

In [1]:
import os

# 👉 Replace with your OpenRouter API key
os.environ["OPENAI_API_KEY"] = "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"

BASE_URL = "https://openrouter.ai/api/v1"

In [ ]:
!pip install langchain-community langchain openai chromadb --quiet

Load Wikipedia Data

In [3]:
from langchain_community.document_loaders import WebBaseLoader
print("Everything Ok")

Everything Ok


Wikipedia installer

In [ ]:
!pip install wikipedia

Wikipedia Loader

In [5]:
from langchain_community.document_loaders import WikipediaLoader

loader = WikipediaLoader(query="Generative artificial intelligence", load_max_docs=2)
documents = loader.load()

print(len(documents))
print(documents[0].page_content[:500])

2
Generative artificial intelligence, also known as generative AI or GenAI, is a subfield of artificial intelligence that uses generative models to generate text, images, videos, audio, software code or other forms of data. These models learn the underlying patterns and structures of their training data, and use them to generate new data in response to input, which often takes the form of natural language prompts.
The prevalence of generative AI tools has increased significantly since the AI boom 


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split documents
chunks = text_splitter.split_documents(documents)

# Check result
print(len(chunks))
print(chunks[0].page_content)

27
Generative artificial intelligence, also known as generative AI or GenAI, is a subfield of artificial intelligence that uses generative models to generate text, images, videos, audio, software code or other forms of data. These models learn the underlying patterns and structures of their training data, and use them to generate new data in response to input, which often takes the form of natural language prompts.


What’s happening
chunk_size=500 → each piece = 500 characters
chunk_overlap=50 → small overlap for better context
Output → many small meaningful text chunks
 Expected output
Number like: 10, 15, etc.
A chunk of text printed
Why this step matters (interview tip )

If interviewer asks:

 “Why chunking in RAG?”

You say:

“To improve retrieval accuracy and fit LLM token limits while preserving context using overlap.”

Test retrieval

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Create embedding model
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create vector database
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding
)

print("Vector DB created successfully")

In [8]:
query = "What is Generative AI?"

results = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(results):
    print(f"\nResult {i+1}:\n")
    print(doc.page_content)


Result 1:

Generative artificial intelligence, also known as generative AI or GenAI, is a subfield of artificial intelligence that uses generative models to generate text, images, videos, audio, software code or other forms of data. These models learn the underlying patterns and structures of their training data, and use them to generate new data in response to input, which often takes the form of natural language prompts.

Result 2:

Companies in a variety of sectors have used generative AI, including those in software development, healthcare, finance, entertainment, customer service, sales and marketing, art, writing, and product design.


Connect LLM (OpenRouter) + Final RAG

Take user question
Retrieve relevant chunks ✅ (you already did)
Send to LLM
Get final intelligent answer

# Take user question
# Retrieve relevant chunks
# Send to LLM
# Get final intelligent answer

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os

# Set OpenRouter API key
os.environ["OPENAI_API_KEY"] = "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"

# LLM
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    model="meta-llama/llama-3-8b-instruct",
    temperature=0.7
)

# Create retriever
retriever = vectorstore.as_retriever()

# Prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below:

{context}

Question: {question}
""")

# Function to run RAG
def rag_pipeline(question):
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    final_prompt = prompt.format(context=context, question=question)

    response = llm.invoke(final_prompt)
    return response.content

# Test
query = "Explain Generative AI in simple terms"
answer = rag_pipeline(query)

print(answer)

Generative AI is a type of artificial intelligence that can create new data, like text, images, or videos, based on patterns and structures it learned from existing data. It can generate new content in response to a prompt or input, like a question or a phrase.


Asking questions

In [17]:
print(rag_pipeline("What are applications of generative AI"))

The applications of generative AI include:

1. Chatbots (e.g. ChatGPT, Claude, Copilot, DeepSeek, Google Gemini, and Grok)
2. Text-to-image models (e.g. Stable Diffusion, Flux, Midjourney, and DALL-E)
3. Text-to-video models (e.g. Veo, LTX, and Sora)
4. Art
5. Writing
6. Software code
7. Product design
8. Customer service
9. Sales and marketing
10. Entertainment


Generative AI models are used to power what products

In [18]:
print(rag_pipeline("Generative AI models are used to power what products"))

According to the context, generative AI models are used to power various products, including:

* Chatbots (e.g. ChatGPT, Claude, Copilot, DeepSeek, Google Gemini, Grok)
* Text-to-image models (e.g. Stable Diffusion, Flux, Midjourney, DALL-E)
* Text-to-video models (e.g. Veo, LTX, Sora)


In [21]:
print(rag_pipeline("Explain in brief Copyright of AI-generated content"))

According to the context, generative AI models have been trained on copyrighted works without the rightholders' permission. This implies that there is a lack of clarity or regulation regarding the copyright of AI-generated content, leaving room for disputes and potential legal issues.
